# 05 - Feature Engineering

**Regra dura: nenhuma feature usa estatistica calculada sobre o conjunto inteiro.**
`build_features(df)` recebe UM DataFrame e devolve o mesmo DataFrame com colunas novas.
Nao recebe, le ou referencia nenhum outro conjunto. E aplicada separadamente a train,
validation, test e transfer_60m.

Nao salva nada em disco. Nao faz commit. So diagnostico e evidencia - a materializacao vem
depois da decisao sobre denominadores zero e sentinelas.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import roc_auc_score

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)

PROCESSED_DIR = Path('..') / 'data' / 'processed'

train = pd.read_parquet(PROCESSED_DIR / 'train.parquet')
validation = pd.read_parquet(PROCESSED_DIR / 'validation.parquet')
test = pd.read_parquet(PROCESSED_DIR / 'test.parquet')
transfer_60m = pd.read_parquet(PROCESSED_DIR / 'transfer_60m.parquet')

print('train:', train.shape)
print('validation:', validation.shape)
print('test:', test.shape)
print('transfer_60m:', transfer_60m.shape)


train: (172988, 84)
validation: (162570, 84)
test: (282787, 84)
transfer_60m: (54969, 84)


## Secao 1 - Diagnostico dos denominadores (TREINO apenas)

Somente leitura, nenhum tratamento aqui - so evidencia para decidir depois.

In [2]:
zero_inc_mask = train['annual_inc'] == 0
low_inc_mask = train['annual_inc'] < 1000

print(f'annual_inc == 0: {int(zero_inc_mask.sum())}')
print(f'annual_inc < 1000: {int(low_inc_mask.sum())}')

cols_show = ['emp_length_anos', 'emp_length_missing', 'home_ownership', 'loan_amnt', 'dti',
             'verification_status', 'target']
print()
print('Linhas com annual_inc < 1000:')
train.loc[low_inc_mask, cols_show]


annual_inc == 0: 0
annual_inc < 1000: 0

Linhas com annual_inc < 1000:


,emp_length_anos,emp_length_missing,home_ownership,loan_amnt,dti,verification_status,target


In [3]:
default_rate_low_inc = train.loc[low_inc_mask, 'target'].mean() * 100
print(f'Taxa de default entre annual_inc < 1000: {default_rate_low_inc:.4f}%')
print(f'(referencia - taxa de default do treino inteiro: {train["target"].mean() * 100:.4f}%)')


Taxa de default entre annual_inc < 1000: nan%
(referencia - taxa de default do treino inteiro: 12.4332%)


In [4]:
print(f'total_acc == 0: {int((train["total_acc"] == 0).sum())}')
print(f'open_acc == 0: {int((train["open_acc"] == 0).sum())}')


total_acc == 0: 0
open_acc == 0: 2


In [5]:
print(f'total_rev_hi_lim == 0: {int((train["total_rev_hi_lim"] == 0).sum())}')
print(f'revol_bal == 0: {int((train["revol_bal"] == 0).sum())}')


total_rev_hi_lim == 0: 56
revol_bal == 0: 1245


In [6]:
missing_or_after = train['earliest_cr_line'].isna() | (train['earliest_cr_line'] > train['issue_d'])
print(f'earliest_cr_line ausente ou posterior a issue_d: {int(missing_or_after.sum())} (esperado: 0)')


earliest_cr_line ausente ou posterior a issue_d: 0 (esperado: 0)


## Secao 2 - Construcao das features (ajustada)

fico_mean e bankcard_to_total_limit foram removidas (ver motivos no docstring de
build_features abaixo). Restam as cinco: installment_to_income, loan_to_income,
credit_history_months, revol_bal_to_income, open_acc_ratio.

Para as divisoes: onde o denominador for zero, o resultado natural da divisao em ponto
flutuante (np.inf) e mantido, sem substituicao.

In [7]:
def build_features(df):
    """Recebe UM DataFrame, devolve o mesmo com colunas novas. So transformacoes row-wise;
    nao le, nao recebe e nao referencia nenhum outro conjunto.

    fico_mean e bankcard_to_total_limit foram removidas: fico_mean tem correlacao 1.0000
    com fico_range_high (spread low/high e constante, media e apenas translacao linear -
    nao adiciona informacao). bankcard_to_total_limit foi descartada por ter ~30% do treino
    sem valor definido (sentinela em era_pre_2012 mais 0/0 organico) em troca de AUC
    univariada de apenas 0.5393."""
    df = df.copy()

    df['installment_to_income'] = df['installment'] / (df['annual_inc'] / 12)
    df['loan_to_income'] = df['loan_amnt'] / df['annual_inc']
    df['credit_history_months'] = ((df['issue_d'].dt.year - df['earliest_cr_line'].dt.year) * 12
                                    + (df['issue_d'].dt.month - df['earliest_cr_line'].dt.month))
    df['revol_bal_to_income'] = df['revol_bal'] / df['annual_inc']
    df['open_acc_ratio'] = df['open_acc'] / df['total_acc']

    return df


new_features = ['installment_to_income', 'loan_to_income', 'credit_history_months',
                 'revol_bal_to_income', 'open_acc_ratio']

train_feat = build_features(train)
validation_feat = build_features(validation)
test_feat = build_features(test)
transfer_60m_feat = build_features(transfer_60m)

sets = {'train': train_feat, 'validation': validation_feat, 'test': test_feat, 'transfer_60m': transfer_60m_feat}
print('build_features aplicada separadamente a cada conjunto. Colunas novas:', new_features)


build_features aplicada separadamente a cada conjunto. Colunas novas: ['installment_to_income', 'loan_to_income', 'credit_history_months', 'revol_bal_to_income', 'open_acc_ratio']


### Contagem de infinitos por feature, em cada conjunto

In [8]:
inf_report = []
for name, d in sets.items():
    row = {'conjunto': name}
    for f in new_features:
        row[f] = int(np.isinf(d[f]).sum())
    inf_report.append(row)

pd.DataFrame(inf_report).set_index('conjunto')


,installment_to_income,loan_to_income,credit_history_months,revol_bal_to_income,open_acc_ratio
conjunto,,,,,
train,0,0,0,0,0
validation,0,0,0,0,0
test,0,0,0,0,0
transfer_60m,0,0,0,0,0


## Secao 3 - Validacao das features novas (TREINO apenas)

### 1. describe() com percentis 1, 25, 50, 75, 99

In [9]:
train_feat[new_features].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99])


,installment_to_income,loan_to_income,credit_history_months,revol_bal_to_income,open_acc_ratio
count,172988.000000,172988.000000,172988.000000,172988.000000,172988.000000
mean,0.078444,0.194426,179.386853,0.228868,0.491985
std,0.042402,0.103569,85.529539,0.180945,0.177495
min,0.000289,0.000789,36.000000,0.000000,0.000000
1%,0.010006,0.025000,47.000000,0.001067,0.166667
25%,0.045598,0.114286,122.000000,0.108721,0.361702
50%,0.072305,0.180000,163.000000,0.191887,0.470588
75%,0.105997,0.263158,221.000000,0.305725,0.600000
99%,0.187648,0.448609,451.000000,0.795986,1.000000
max,0.320262,0.830000,785.000000,6.086388,1.750000


### 2. AUC univariada contra o target (infinitos excluidos desta checagem, tratamento pendente)

In [10]:
auc_rows = []
for f in new_features:
    s = train_feat[[f, 'target']].replace([np.inf, -np.inf], np.nan).dropna()
    if s[f].nunique() < 2:
        auc_rows.append({'feature': f, 'n_validos': len(s), 'auc': np.nan, 'auc_max_dir': np.nan})
        continue
    auc = roc_auc_score(s['target'], s[f])
    auc_rows.append({'feature': f, 'n_validos': len(s), 'auc': round(auc, 4), 'auc_max_dir': round(max(auc, 1 - auc), 4)})

pd.DataFrame(auc_rows)


,feature,n_validos,auc,auc_max_dir
0,installment_to_income,172988,0.5709,0.5709
1,loan_to_income,172988,0.5588,0.5588
2,credit_history_months,172988,0.4626,0.5374
3,revol_bal_to_income,172988,0.5327,0.5327
4,open_acc_ratio,172988,0.5418,0.5418


### 3. Correlacao de Pearson com as features existentes mais proximas

In [11]:
pairs = [
    ('installment_to_income', 'dti'),
    ('loan_to_income', 'dti'),
    ('revol_bal_to_income', 'revol_util'),
    ('credit_history_months', 'mo_sin_old_rev_tl_op'),
]

corr_rows = []
for f1, f2 in pairs:
    s = train_feat[[f1, f2]].replace([np.inf, -np.inf], np.nan).dropna()
    corr = s[f1].corr(s[f2], method='pearson')
    flag = 'REDUNDANCIA (|r| > 0.90)' if abs(corr) > 0.90 else ''
    corr_rows.append({'feature_nova': f1, 'feature_existente': f2, 'n_validos': len(s),
                       'pearson_r': round(corr, 4), 'sinalizacao': flag})

pd.DataFrame(corr_rows)


,feature_nova,feature_existente,n_validos,pearson_r,sinalizacao
0,installment_to_income,dti,172988,0.2376,
1,loan_to_income,dti,172988,0.2239,
2,revol_bal_to_income,revol_util,172988,0.2921,
3,credit_history_months,mo_sin_old_rev_tl_op,172988,0.6181,


### 4. Taxa de default por decil de cada feature nova (checagem de monotonicidade)

In [12]:
for f in new_features:
    s = train_feat[[f, 'target']].replace([np.inf, -np.inf], np.nan).dropna()
    try:
        s = s.copy()
        s['decil'] = pd.qcut(s[f], 10, labels=False, duplicates='drop')
    except ValueError as e:
        print(f'--- {f} ---')
        print(f'  Nao foi possivel formar decis distintos: {e}')
        print()
        continue

    decile_table = s.groupby('decil').agg(n=('target', 'count'), min_val=(f, 'min'),
                                            max_val=(f, 'max'), taxa_default=('target', 'mean'))
    decile_table['taxa_default'] = (decile_table['taxa_default'] * 100).round(2)
    print(f'--- {f} ---')
    display(decile_table)


--- installment_to_income ---


,n,min_val,max_val,taxa_default
decil,,,,
0,17299,0.000289,0.027565,9.52
1,17299,0.027566,0.039997,10.02
2,17299,0.039998,0.050815,9.84
3,17298,0.050815,0.061422,10.51
4,17299,0.061423,0.072305,11.16
5,17300,0.072305,0.083998,12.23
6,17297,0.084000,0.097903,12.29
7,17299,0.097905,0.115166,13.99
8,17299,0.115169,0.138571,16.10


--- loan_to_income ---


,n,min_val,max_val,taxa_default
decil,,,,
0,17321,0.000789,0.068966,9.90
1,18095,0.068997,0.100000,10.46
2,16481,0.100000,0.127174,10.19
3,17963,0.127176,0.153846,10.81
4,16843,0.153846,0.180000,11.55
5,17276,0.180002,0.208333,12.44
6,17157,0.208420,0.242857,12.07
7,17275,0.242866,0.285000,14.13
8,17281,0.285008,0.339623,15.79


--- credit_history_months ---


,n,min_val,max_val,taxa_default
decil,,,,
0,17679,36,83,15.55
1,17213,84,110,13.27
2,17526,111,131,13.47
3,17304,132,147,13.03
4,17233,148,163,12.58
5,17306,164,182,12.51
6,17292,183,207,11.77
7,17129,208,239,10.80
8,17083,240,295,10.74


--- revol_bal_to_income ---


,n,min_val,max_val,taxa_default
decil,,,,
0,17299,0.000000,0.052673,10.98
1,17299,0.052673,0.091933,11.20
2,17299,0.091936,0.124838,11.20
3,17298,0.124838,0.157678,11.45
4,17299,0.157681,0.191885,11.78
5,17299,0.191889,0.230485,12.53
6,17298,0.230486,0.277400,13.00
7,17299,0.277404,0.339660,13.10
8,17299,0.339662,0.443452,14.27


--- open_acc_ratio ---


,n,min_val,max_val,taxa_default
decil,,,,
0,17310,0.000000,0.279070,10.71
1,18284,0.279412,0.333333,10.73
2,17224,0.337838,0.384615,10.66
3,18311,0.385965,0.428571,11.73
4,16080,0.431034,0.470588,11.74
5,16874,0.471698,0.518519,12.37
6,18221,0.519231,0.571429,12.63
7,16223,0.574468,0.636364,13.56
8,17302,0.638298,0.739130,14.86


## Secao 4 - Comparacao entre conjuntos (SMD treino vs. teste)

Mesma metodologia da Secao 3 do notebook 04. Infinitos excluidos (tratamento pendente).
Sinaliza SMD > 0.25.

In [13]:
smd_rows = []
for f in new_features:
    s_train = train_feat[f].replace([np.inf, -np.inf], np.nan).dropna()
    s_test = test_feat[f].replace([np.inf, -np.inf], np.nan).dropna()
    mean_train, mean_test = s_train.mean(), s_test.mean()
    std_train, std_test = s_train.std(), s_test.std()
    pooled_std = np.sqrt((std_train ** 2 + std_test ** 2) / 2)
    smd = abs(mean_train - mean_test) / pooled_std if pooled_std > 0 else 0.0
    flag = 'SINALIZADO (SMD > 0.25)' if smd > 0.25 else ''
    smd_rows.append({'feature': f, 'n_treino': len(s_train), 'n_teste': len(s_test),
                      'media_treino': mean_train, 'media_teste': mean_test,
                      'smd': round(smd, 4), 'sinalizacao': flag})

pd.DataFrame(smd_rows).sort_values('smd', ascending=False)


,feature,n_treino,n_teste,media_treino,media_teste,smd,sinalizacao
2,credit_history_months,172988,282787,179.386853,199.101345,0.2202,
4,open_acc_ratio,172988,282787,0.491985,0.510873,0.1056,
1,loan_to_income,172988,282787,0.194426,0.196912,0.0236,
0,installment_to_income,172988,282787,0.078444,0.078049,0.0092,
3,revol_bal_to_income,172988,282787,0.228868,0.229926,0.0053,


## Secao 5 - FEATURE_SET e EXCLUDED

In [14]:
EVAL_ONLY = ['loan_status', 'loan_amnt', 'installment', 'term', 'total_rec_prncp']
PROVISIONAL_EXCLUDE = ['int_rate', 'grade', 'sub_grade']

family_C_features = ['funded_amnt', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d',
    'purpose', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'fico_range_low', 'fico_range_high',
    'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec',
    'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'collections_12_mths_ex_med',
    'mths_since_last_major_derog', 'application_type', 'acc_now_delinq', 'tot_coll_amt',
    'tot_cur_bal', 'total_rev_hi_lim', 'acc_open_past_24mths', 'avg_cur_bal', 'bc_open_to_buy',
    'bc_util', 'chargeoff_within_12_mths', 'delinq_amnt', 'mo_sin_old_il_acct',
    'mo_sin_old_rev_tl_op', 'mo_sin_rcnt_rev_tl_op', 'mo_sin_rcnt_tl', 'mort_acc',
    'mths_since_recent_bc', 'mths_since_recent_bc_dlq', 'mths_since_recent_inq',
    'mths_since_recent_revol_delinq', 'num_accts_ever_120_pd', 'num_actv_bc_tl', 'num_actv_rev_tl',
    'num_bc_sats', 'num_bc_tl', 'num_il_tl', 'num_op_rev_tl', 'num_rev_accts',
    'num_rev_tl_bal_gt_0', 'num_sats', 'num_tl_120dpd_2m', 'num_tl_30dpd', 'num_tl_90g_dpd_24m',
    'num_tl_op_past_12m', 'pct_tl_nvr_dlq', 'percent_bc_gt_75', 'pub_rec_bankruptcies',
    'tax_liens', 'tot_hi_cred_lim', 'total_bal_ex_mort', 'total_bc_limit',
    'total_il_high_credit_limit', 'emp_length_anos']
assert len(family_C_features) == 65

engineered_flags = ['era_pre_2012',
                     'mths_since_last_delinq_missing', 'mths_since_last_record_missing',
                     'mths_since_recent_bc_dlq_missing', 'mths_since_recent_revol_delinq_missing',
                     'mths_since_last_major_derog_missing', 'emp_length_missing',
                     'mths_since_recent_inq_missing', 'num_tl_120dpd_2m_missing', 'sparse_bureau_missing']
assert len(engineered_flags) == 10

redundant_cols = {'fico_range_high': 'redundancia (r=1.0 com fico_range_low - spread constante)'}

FEATURE_SET = [c for c in family_C_features if c not in redundant_cols] + engineered_flags + new_features

EXCLUDED = []
for c in EVAL_ONLY:
    EXCLUDED.append({'coluna': c, 'motivo': 'EVAL_ONLY'})
for c in PROVISIONAL_EXCLUDE:
    EXCLUDED.append({'coluna': c, 'motivo': 'PROVISIONAL_EXCLUDE'})
for c, motivo in redundant_cols.items():
    EXCLUDED.append({'coluna': c, 'motivo': motivo})
# texto livre (emp_title, title, desc, zip_code): ja descartadas do parquet em 03_build_processed.ipynb,
# nao ha coluna dessa categoria remanescente para excluir aqui.

print(f'FEATURE_SET: {len(FEATURE_SET)} colunas')
print(FEATURE_SET)
print()
print(f'EXCLUDED: {len(EXCLUDED)} colunas')
for item in EXCLUDED:
    print(f"  {item['coluna']}: {item['motivo']}")
print()
print("'target' (o label) e as colunas ja descartadas do parquet (familia A/B/D) nao entram em")
print('nenhuma das duas listas - nao sao features nem estao mais presentes no dataset.')


FEATURE_SET: 79 colunas
['funded_amnt', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d', 'purpose', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'fico_range_low', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'collections_12_mths_ex_med', 'mths_since_last_major_derog', 'application_type', 'acc_now_delinq', 'tot_coll_amt', 'tot_cur_bal', 'total_rev_hi_lim', 'acc_open_past_24mths', 'avg_cur_bal', 'bc_open_to_buy', 'bc_util', 'chargeoff_within_12_mths', 'delinq_amnt', 'mo_sin_old_il_acct', 'mo_sin_old_rev_tl_op', 'mo_sin_rcnt_rev_tl_op', 'mo_sin_rcnt_tl', 'mort_acc', 'mths_since_recent_bc', 'mths_since_recent_bc_dlq', 'mths_since_recent_inq', 'mths_since_recent_revol_delinq', 'num_accts_ever_120_pd', 'num_actv_bc_tl', 'num_actv_rev_tl', 'num_bc_sats', 'num_bc_tl', 'num_il_tl', 'num_op_rev_tl', 'num_rev_accts', 'num_rev_tl_bal_gt_0', 'num_sats', 'num_tl_120dpd_2m', 'n

## Secao 6 - Materializacao

Aplica `build_features` separadamente aos quatro conjuntos e sobrescreve os parquets.

In [15]:
shapes_before = {name: d.shape for name, d in [('train', train), ('validation', validation),
                                                  ('test', test), ('transfer_60m', transfer_60m)]}

train_final = build_features(train)
validation_final = build_features(validation)
test_final = build_features(test)
transfer_60m_final = build_features(transfer_60m)

final_sets = {'train': train_final, 'validation': validation_final,
              'test': test_final, 'transfer_60m': transfer_60m_final}

train_final.to_parquet(PROCESSED_DIR / 'train.parquet', index=False)
validation_final.to_parquet(PROCESSED_DIR / 'validation.parquet', index=False)
test_final.to_parquet(PROCESSED_DIR / 'test.parquet', index=False)
transfer_60m_final.to_parquet(PROCESSED_DIR / 'transfer_60m.parquet', index=False)

for name, d in final_sets.items():
    print(f'{name}: antes={shapes_before[name]} -> depois={d.shape}')


train: antes=(172988, 84) -> depois=(172988, 89)
validation: antes=(162570, 84) -> depois=(162570, 89)
test: antes=(282787, 84) -> depois=(282787, 89)
transfer_60m: antes=(54969, 84) -> depois=(54969, 89)


## Secao 7 - Verificacao final

In [16]:
problems = []
for name, d in final_sets.items():
    for f in new_features:
        n_nan = int(d[f].isna().sum())
        n_inf = int(np.isinf(d[f]).sum())
        if n_nan > 0 or n_inf > 0:
            problems.append({'conjunto': name, 'feature': f, 'n_nan': n_nan, 'n_inf': n_inf})

if problems:
    print('PROBLEMAS ENCONTRADOS:')
    for p in problems:
        print(f"  {p['conjunto']} / {p['feature']}: NaN={p['n_nan']} inf={p['n_inf']}")
    raise AssertionError('PARANDO: as 5 features novas devem estar completamente definidas (sem NaN/inf).')
else:
    print('Nenhuma das 5 features novas tem NaN ou infinito, em nenhum dos quatro conjuntos. OK.')


Nenhuma das 5 features novas tem NaN ou infinito, em nenhum dos quatro conjuntos. OK.


In [17]:
overlap_eval = set(FEATURE_SET) & set(EVAL_ONLY)
overlap_prov = set(FEATURE_SET) & set(PROVISIONAL_EXCLUDE)

print(f'FEATURE_SET intersecta EVAL_ONLY? {overlap_eval if overlap_eval else "Nao."}')
print(f'FEATURE_SET intersecta PROVISIONAL_EXCLUDE? {overlap_prov if overlap_prov else "Nao."}')

assert not overlap_eval, f'PARANDO: FEATURE_SET contem coluna(s) de EVAL_ONLY: {overlap_eval}'
assert not overlap_prov, f'PARANDO: FEATURE_SET contem coluna(s) de PROVISIONAL_EXCLUDE: {overlap_prov}'

print()
print(f'Contagem final de features em FEATURE_SET: {len(FEATURE_SET)}')


FEATURE_SET intersecta EVAL_ONLY? Nao.
FEATURE_SET intersecta PROVISIONAL_EXCLUDE? Nao.

Contagem final de features em FEATURE_SET: 79
